In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('iris.csv')
X = df.iloc[:, 1:-1].values  # Drop ID and Species columns
y = df.iloc[:, -1].values

In [ ]:
# Split data into 80% train and 20% test manually
indices = np.random.permutation(len(X))
test_size = int(len(X) * 0.2)
train_idx, test_idx = indices[test_size:], indices[:test_size]
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

In [1]:
print(y)

NameError: name 'y' is not defined

In [ ]:
class NaiveBayesScratch:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.parameters = []
        for i, c in enumerate(self.classes):
            X_c = X[y == c]
            self.parameters.append([])
            for feature in range(X.shape[1]):
                parameters = {"mean": X_c[:, feature].mean(), "var": X_c[:, feature].var()}
                self.parameters[i].append(parameters)

    def _calculate_likelihood(self, class_idx, feature_idx, x):
        mean = self.parameters[class_idx][feature_idx]["mean"]
        var = self.parameters[class_idx][feature_idx]["var"]
        numerator = np.exp(- (x - mean)**2 / (2 * var))
        denominator = np.sqrt(2 * np.pi * var)
        return numerator / denominator

    def predict(self, X):
        y_pred = []
        for x in X:
            posteriors = []
            for i, c in enumerate(self.classes):
                # Prior P(y)
                prior = np.log(len(y_train[y_train == c]) / len(y_train))
                # Log likelihood to avoid underflow
                likelihood = np.sum([np.log(self._calculate_likelihood(i, j, x[j])) for j in range(len(x))])
                posteriors.append(prior + likelihood)
            y_pred.append(self.classes[np.argmax(posteriors)])
        return np.array(y_pred)

In [ ]:
model = NaiveBayesScratch()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

In [ ]:
# 3. Compute Metrics (Manual Implementation)
def get_metrics(y_true, y_pred):
    classes = np.unique(y_true)
    # For multiclass, we'll calculate metrics for one class (e.g., Iris-setosa) vs others
    # or just use the overall accuracy. Let's look at Iris-setosa specifically for TP/FP:
    target = classes[0]

    tp = np.sum((y_true == target) & (y_pred == target))
    tn = np.sum((y_true != target) & (y_pred != target))
    fp = np.sum((y_true != target) & (y_pred == target))
    fn = np.sum((y_true == target) & (y_pred != target))

    accuracy = np.sum(y_true == y_pred) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    error_rate = 1 - accuracy

    return tp, fp, tn, fn, accuracy, precision, recall, error_rate

tp, fp, tn, fn, acc, prec, rec, err = get_metrics(y_test, predictions)

In [ ]:
# Output Results
print(f"--- Results for Class: {np.unique(y_test)[0]} ---")
print(f"Confusion Matrix: TP={tp}, FP={fp}, TN={tn}, FN={fn}")
print(f"Accuracy:   {acc:.2%}")
print(f"Error Rate: {err:.2%}")
print(f"Precision:  {prec:.2f}")
print(f"Recall:     {rec:.2f}")

--- Results for Class: Iris-setosa ---
Confusion Matrix: TP=10, FP=0, TN=20, FN=0
Accuracy:   96.67%
Error Rate: 3.33%
Precision:  1.00
Recall:     1.00


In [ ]:
def get_all_metrics(y_true, y_pred):
    classes = np.unique(y_true)
    results = []

    for cls in classes:
        # TP: Predicted class and is actually class
        tp = np.sum((y_true == cls) & (y_pred == cls))
        # TN: Predicted NOT class and is actually NOT class
        tn = np.sum((y_true != cls) & (y_pred != cls))
        # FP: Predicted class but is actually NOT class
        fp = np.sum((y_true != cls) & (y_pred == cls))
        # FN: Predicted NOT class but is actually class
        fn = np.sum((y_true == cls) & (y_pred != cls))

        accuracy = (tp + tn) / len(y_true)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        error_rate = 1 - accuracy

        results.append({
            "Species": cls,
            "TP": tp, "FP": fp, "TN": tn, "FN": fn,
            "Precision": round(precision, 2),
            "Recall": round(recall, 2),
            "Accuracy": round(accuracy, 2)
        })

    return pd.DataFrame(results)

# Assuming 'predictions' and 'y_test' come from the previous code
stats_df = get_all_metrics(y_test, predictions)
print(stats_df)

           Species  TP  FP  TN  FN  Precision  Recall  Accuracy
0      Iris-setosa  10   0  20   0       1.00     1.0      1.00
1  Iris-versicolor  10   1  19   0       0.91     1.0      0.97
2   Iris-virginica   9   0  20   1       1.00     0.9      0.97


In [ ]:
def compute_confusion_matrix(y_true, y_pred):
    classes = np.unique(y_true)
    num_classes = len(classes)

    # Initialize a 3x3 matrix with zeros
    matrix = np.zeros((num_classes, num_classes), dtype=int)

    # Map class names to indices (0, 1, 2)
    class_to_idx = {cls: i for i, cls in enumerate(classes)}

    # Fill the matrix
    # Rows = Actual, Columns = Predicted
    for actual, predicted in zip(y_true, y_pred):
        i = class_to_idx[actual]
        j = class_to_idx[predicted]
        matrix[i, j] += 1

    # Convert to DataFrame for better visualization
    matrix_df = pd.DataFrame(matrix, index=classes, columns=classes)
    return matrix_df

# Generate the matrix
conf_matrix = compute_confusion_matrix(y_test, predictions)

print("--- 3x3 Confusion Matrix ---")
print("Rows = Actual Classes | Columns = Predicted Classes")
print(conf_matrix)

--- 3x3 Confusion Matrix ---
Rows = Actual Classes | Columns = Predicted Classes
                 Iris-setosa  Iris-versicolor  Iris-virginica
Iris-setosa               10                0               0
Iris-versicolor            0               10               0
Iris-virginica             0                1               9
